# JSON to Pandas Converter for X Scraper Data

This notebook converts JSON output from the X/Twitter scraper into pandas DataFrames for analysis.

Upload your JSON files or use the output from the scraper notebook.

In [ ]:
# Install required packages
!pip install pandas matplotlib seaborn

import pandas as pd
import json
from pathlib import Path
from typing import Dict, List, Any, Optional, Union
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("✅ Setup complete!")

## Upload JSON Files

Upload your JSON files from the scraper:

In [ ]:
# Upload JSON files
from google.colab import files

uploaded = files.upload()

uploaded_files = []
for filename in uploaded.keys():
    print(f'✅ Uploaded: {filename}')
    uploaded_files.append(filename)

## Data Conversion Functions

In [ ]:
class PandasConverter:
    """Converter for JSON scraper output to pandas DataFrames"""
    
    def load_json_file(self, filepath: str) -> Union[Dict, List, None]:
        """Load a JSON file and return the data"""
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"❌ Error loading {filepath}: {e}")
            return None
    
    def convert_tweets_to_dataframe(self, tweets_data: List[Dict]) -> pd.DataFrame:
        """Convert tweets data to a pandas DataFrame"""
        if not tweets_data:
            return pd.DataFrame()
        
        df = pd.DataFrame(tweets_data)
        
        # Convert date strings to datetime
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
        
        # Flatten nested structures
        if 'hashtags' in df.columns:
            df['hashtags_count'] = df['hashtags'].apply(lambda x: len(x) if isinstance(x, list) else 0)
            df['hashtags_str'] = df['hashtags'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
        
        if 'mentioned_users' in df.columns:
            df['mentioned_users_count'] = df['mentioned_users'].apply(lambda x: len(x) if isinstance(x, list) else 0)
        
        if 'media' in df.columns:
            df['media_count'] = df['media'].apply(lambda x: len(x) if isinstance(x, list) else 0)
            df['has_media'] = df['media_count'] > 0
        
        # Add engagement score
        df['engagement_score'] = (
            df.get('like_count', 0) + 
            df.get('retweet_count', 0) + 
            df.get('reply_count', 0) + 
            df.get('quote_count', 0)
        )
        
        return df
    
    def convert_user_info_to_dataframe(self, user_data: Dict) -> pd.DataFrame:
        """Convert user info data to a pandas DataFrame"""
        if not user_data:
            return pd.DataFrame()
        
        df = pd.DataFrame([user_data])
        
        if 'joined_date' in df.columns:
            df['joined_date'] = pd.to_datetime(df['joined_date'], errors='coerce')
        
        df['account_age_days'] = (pd.Timestamp.now() - df['joined_date']).dt.days
        df['follower_following_ratio'] = df['followers_count'] / df['following_count'].replace(0, 1)
        
        return df
    
    def convert_followers_to_dataframe(self, followers_data: List[Dict]) -> pd.DataFrame:
        """Convert followers data to a pandas DataFrame"""
        if not followers_data:
            return pd.DataFrame()
        
        df = pd.DataFrame(followers_data)
        df['follower_following_ratio'] = df['followers_count'] / df['following_count'].replace(0, 1)
        
        return df
    
    def detect_data_type(self, data: Union[Dict, List]) -> str:
        """Detect the type of data in the JSON file"""
        if isinstance(data, list) and data:
            if any(key in data[0] for key in ['content', 'rawContent', 'like_count']):
                return 'tweets'
            elif any(key in data[0] for key in ['username', 'followers_count']) and 'content' not in data[0]:
                return 'followers'
        elif isinstance(data, dict):
            if 'followers_count' in data and 'statuses_count' in data:
                return 'user_info'
        
        return 'unknown'
    
    def convert_file(self, filename: str) -> Optional[pd.DataFrame]:
        """Convert a JSON file to a pandas DataFrame"""
        data = self.load_json_file(filename)
        if data is None:
            return None
        
        data_type = self.detect_data_type(data)
        print(f"📄 Processing {filename} (type: {data_type})")
        
        if data_type == 'tweets':
            return self.convert_tweets_to_dataframe(data)
        elif data_type == 'user_info':
            return self.convert_user_info_to_dataframe(data)
        elif data_type == 'followers':
            return self.convert_followers_to_dataframe(data)
        else:
            return pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame([data])

# Initialize converter
converter = PandasConverter()

## Convert Files to DataFrames

In [ ]:
# Convert uploaded files
dataframes = {}

for filename in uploaded_files:
    if filename.endswith('.json'):
        df = converter.convert_file(filename)
        if df is not None:
            df_name = Path(filename).stem
            dataframes[df_name] = df
            print(f"✅ Created DataFrame: {df_name} (shape: {df.shape})")
            # Make it available globally
            globals()[f'df_{df_name}'] = df

print(f"\n📊 Available DataFrames:")
for name, df in dataframes.items():
    print(f"  - df_{name}: {df.shape}")

## Data Analysis

Choose a DataFrame to analyze:

In [ ]:
#@title 📊 Analyze DataFrame

analyze_dataframe = "search_AI_technology"  #@param {type:"string"}
show_preview = True  #@param {type:"boolean"}
show_stats = True  #@param {type:"boolean"}
show_plots = False  #@param {type:"boolean"}

if analyze_dataframe in dataframes:
    df = dataframes[analyze_dataframe]
    
    print(f"📊 Analyzing: {analyze_dataframe}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    if show_preview:
        print("\n🔍 Preview:")
        display(df.head())
    
    if show_stats:
        print("\n📈 Statistics:")
        display(df.describe())
        
        # Show data types
        print("\n📋 Data Types:")
        print(df.dtypes)
        
        # Show missing values
        print("\n❓ Missing Values:")
        missing = df.isnull().sum()
        if missing.sum() > 0:
            print(missing[missing > 0])
        else:
            print("No missing values found")
    
    if show_plots and len(df) > 0:
        print("\n📊 Plots:")
        
        # Check for numeric columns to plot
        numeric_cols = df.select_dtypes(include=['number']).columns
        
        if len(numeric_cols) > 0:
            # Engagement metrics plot
            engagement_cols = [col for col in ['like_count', 'retweet_count', 'reply_count', 'quote_count'] if col in numeric_cols]
            if engagement_cols:
                plt.figure(figsize=(12, 6))
                df[engagement_cols].boxplot()
                plt.title('Engagement Metrics Distribution')
                plt.ylabel('Count')
                plt.xticks(rotation=45)
                plt.show()
            
            # Correlation heatmap
            if len(numeric_cols) > 2:
                plt.figure(figsize=(10, 8))
                correlation_matrix = df[numeric_cols].corr()
                sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
                plt.title('Correlation Matrix')
                plt.show()
        
        # Time series if date column exists
        if 'date' in df.columns and pd.api.types.is_datetime64_any_dtype(df['date']):
            df_time = df.set_index('date').sort_index()
            if 'engagement_score' in df_time.columns:
                plt.figure(figsize=(14, 6))
                df_time['engagement_score'].plot()
                plt.title('Engagement Score Over Time')
                plt.ylabel('Engagement Score')
                plt.show()
else:
    print(f"❌ DataFrame '{analyze_dataframe}' not found.")
    print("Available DataFrames:")
    for name in dataframes.keys():
        print(f"  - {name}")

## Export to CSV

In [ ]:
#@title 💾 Export DataFrames to CSV

export_all = True  #@param {type:"boolean"}
specific_dataframe = ""  #@param {type:"string"}

from google.colab import files

if export_all:
    for name, df in dataframes.items():
        filename = f"{name}.csv"
        df.to_csv(filename, index=False, encoding='utf-8')
        print(f"💾 Exported: {filename}")
        files.download(filename)
elif specific_dataframe and specific_dataframe in dataframes:
    df = dataframes[specific_dataframe]
    filename = f"{specific_dataframe}.csv"
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"💾 Exported: {filename}")
    files.download(filename)
else:
    print("⚙️ Configure export settings above")

## Custom Analysis

Add your own analysis code here:

In [ ]:
# Custom analysis space
# Use the DataFrames created above (e.g., df_search_AI_technology, df_user_elonmusk, etc.)

# Example: Find top tweets by engagement
# if 'df_search_AI_technology' in globals():
#     top_tweets = df_search_AI_technology.nlargest(10, 'engagement_score')
#     display(top_tweets[['username', 'content', 'engagement_score', 'like_count', 'retweet_count']])

print("💡 Add your custom analysis code here!")
print("Available DataFrames:", list(dataframes.keys()))